<a href="https://colab.research.google.com/github/sakeththelu/NLP/blob/main/4082_nlp_ass14_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Environment Setup

First, let's install the necessary libraries for data manipulation and PyTorch.

In [1]:
import sys
!{sys.executable} -m pip install torch pandas scikit-learn

## Data Loading: SMS Spam Collection Dataset

I will now download and load the SMS Spam Collection dataset. This dataset consists of SMS messages labeled as either 'ham' (legitimate) or 'spam'.

## Text Preprocessing

This step involves cleaning the text data to prepare it for model training. Common preprocessing steps include converting text to lowercase, removing punctuation, and tokenization. We'll start with these basic steps.

In [4]:
import string

def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df['processed_message'] = df['message'].apply(preprocess_text)

print("Text preprocessing complete. Displaying first 5 processed messages:")
display(df[['message', 'processed_message']].head())

Text preprocessing complete. Displaying first 5 processed messages:


,message,processed_message
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


## Vocabulary Creation & Encoding

To prepare the text for a neural network, we need to convert words into numerical representations. This involves:
1. Tokenizing the processed messages into individual words.
2. Building a vocabulary of unique words.
3. Mapping each word to a unique integer index.
4. Encoding the messages into sequences of these integer indices.

In [5]:
from collections import Counter

# Tokenize messages
df['tokens'] = df['processed_message'].apply(lambda x: x.split())

# Build vocabulary
all_words = [word for tokens in df['tokens'] for word in tokens]
word_counts = Counter(all_words)

# Create word-to-index and index-to-word mappings
vocabulary = sorted(word_counts, key=word_counts.get, reverse=True)
# Add a special token for padding (0) and unknown words (1)
word_to_idx = {'<pad>': 0, '<unk>': 1}
for idx, word in enumerate(vocabulary, start=2):
    word_to_idx[word] = idx

idx_to_word = {idx: word for word, idx in word_to_idx.items()}

vocab_size = len(word_to_idx)
print(f"Vocabulary size: {vocab_size}")

# Encode messages
def encode_message(tokens, word_to_idx):
    return [word_to_idx.get(word, word_to_idx['<unk>']) for word in tokens]

df['encoded_messages'] = df['tokens'].apply(lambda x: encode_message(x, word_to_idx))

print("\nFirst 5 original messages, tokens, and encoded messages:")
for i in range(5):
    print(f"Original: {df['message'].iloc[i]}")
    print(f"Tokens: {df['tokens'].iloc[i]}")
    print(f"Encoded: {df['encoded_messages'].iloc[i]}\n")

# Display the DataFrame with new columns
display(df.head())

Vocabulary size: 9663

First 5 original messages, tokens, and encoded messages:
Original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Tokens: ['go', 'until', 'jurong', 'point', 'crazy', 'available', 'only', 'in', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'there', 'got', 'amore', 'wat']
Encoded: [47, 442, 4395, 797, 714, 680, 65, 10, 1252, 91, 122, 356, 1253, 154, 2910, 1254, 69, 58, 4396, 138]

Original: Ok lar... Joking wif u oni...
Tokens: ['ok', 'lar', 'joking', 'wif', 'u', 'oni']
Encoded: [48, 312, 1407, 443, 7, 1843]

Original: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
Tokens: ['free', 'entry', 'in', '2', 'a', 'wkly', 'comp', 'to', 'win', 'fa', 'cup', 'final', 'tkts', '21st', 'may', '2005', 'text', 'fa', 'to', '87121', 'to', 'receive', 'entry', 'questionstd', 'txt', 'ratetcs', 'apply

,label,message,processed_message,tokens,encoded_messages
0,ham,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o...","[47, 442, 4395, 797, 714, 680, 65, 10, 1252, 9..."
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]","[48, 312, 1407, 443, 7, 1843]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[51, 461, 10, 22, 5, 752, 905, 2, 180, 1844, 1..."
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, t...","[7, 233, 142, 25, 357, 2914, 7, 162, 145, 61, ..."
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,"[nah, i, dont, think, he, goes, to, usf, he, l...","[958, 3, 45, 100, 73, 462, 2, 906, 73, 1847, 2..."


## Train-Test Split

We need to split our dataset into training and testing sets to evaluate the model's performance on unseen data. We'll use `scikit-learn`'s `train_test_split` function for this.

In [6]:
from sklearn.model_selection import train_test_split
import numpy as np

# Pad sequences to a fixed length for batch processing
def pad_sequences(sequences, max_len, padding_value=0):
    padded = np.full((len(sequences), max_len), padding_value, dtype=np.int64)
    for i, seq in enumerate(sequences):
        length = min(len(seq), max_len)
        padded[i, :length] = seq[:length]
    return padded

# Determine a reasonable maximum sequence length
# For simplicity, let's use the 90th percentile of message lengths
# or a fixed value if messages are very long. Average message length is around 15 words.
# Let's target a max length around 50 words which should cover most messages.
all_lengths = [len(seq) for seq in df['encoded_messages']]
max_sequence_length = int(np.percentile(all_lengths, 95)) # Use 95th percentile for max_len
print(f"Using max sequence length: {max_sequence_length}")

# Pad encoded messages
X = pad_sequences(df['encoded_messages'].tolist(), max_len=max_sequence_length)

# Convert labels to numerical format (0 for ham, 1 for spam)
y = df['label'].apply(lambda x: 1 if x == 'spam' else 0).values

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("\nTrain-test split complete.")

Using max sequence length: 32

Shape of X_train: (4457, 32)
Shape of X_test: (1115, 32)
Shape of y_train: (4457,)
Shape of y_test: (1115,)

Train-test split complete.


## Create PyTorch Dataset Class

To effectively use our data with PyTorch, we need to define a custom `Dataset` class. This class will handle retrieving our encoded messages and their corresponding labels. We'll also create `DataLoader` instances to efficiently batch and shuffle the data during training.

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class SMSDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Create Dataset instances
train_dataset = SMSDataset(X_train, y_train)
test_dataset = SMSDataset(X_test, y_test)

# Define batch size
batch_size = 64

# Create DataLoader instances
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("PyTorch Dataset and DataLoaders created successfully.")
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")

# Example of one batch
for features, labels in train_loader:
    print(f"\nShape of features in a batch: {features.shape}") # (batch_size, max_sequence_length)
    print(f"Shape of labels in a batch: {labels.shape}")     # (batch_size)
    break

PyTorch Dataset and DataLoaders created successfully.
Number of training batches: 70
Number of testing batches: 18

Shape of features in a batch: torch.Size([64, 32])
Shape of labels in a batch: torch.Size([64])


## Build 1D CNN Model

Now, let's define our 1D Convolutional Neural Network (CNN) model using PyTorch. This model will typically consist of an embedding layer, followed by convolutional layers, pooling layers, and fully connected layers for classification.

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_filters, filter_sizes, output_dim, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])

        self.fc = nn.Linear(len(filter_sizes) * num_filters, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # text = [batch size, sent len]

        embedded = self.embedding(text)

        # embedded = [batch size, sent len, emb dim]

        embedded = embedded.permute(0, 2, 1)

        # embedded = [batch size, emb dim, sent len]

        conved = [F.relu(conv(embedded)) for conv in self.convs]

        # conved_n = [batch size, num filters, (sent len - filter_sizes[n]) + 1]

        pooled = [F.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]

        # pooled_n = [batch size, num filters]

        cat = self.dropout(torch.cat(pooled, dim=1))

        # cat = [batch size, num filters * len(filter_sizes)]

        return self.fc(cat)

# Model Hyperparameters
EMBEDDING_DIM = 100
NUM_FILTERS = 100
FILTER_SIZES = [3, 4, 5] # Tri-gram, 4-gram, 5-gram convolutions
OUTPUT_DIM = 1 # Binary classification (spam or ham)
DROPOUT = 0.5

# Instantiate the model
model = TextCNN(vocab_size, EMBEDDING_DIM, NUM_FILTERS, FILTER_SIZES, OUTPUT_DIM, DROPOUT)

print("1D CNN Model built successfully:")
print(model)

# Function to count trainable parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')


1D CNN Model built successfully:
TextCNN(
  (embedding): Embedding(9663, 100)
  (convs): ModuleList(
    (0): Conv1d(100, 100, kernel_size=(3,), stride=(1,))
    (1): Conv1d(100, 100, kernel_size=(4,), stride=(1,))
    (2): Conv1d(100, 100, kernel_size=(5,), stride=(1,))
  )
  (fc): Linear(in_features=300, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)
The model has 1,086,901 trainable parameters


## Model Training & Model Evaluation

Now we will define the training and evaluation loops, set up the optimizer and loss function, and then train our 1D CNN model. Finally, we'll evaluate its performance on the test dataset.

In [9]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Set device for training (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Optimizer and Loss Function
optimizer = optim.Adam(model.parameters())
criterion = nn.BCEWithLogitsLoss()

criterion = criterion.to(device)

def binary_accuracy(preds, y):
    # Round predictions to the closest integer (0 or 1)
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float() # Convert to float for division
    acc = correct.sum() / len(correct)
    return acc

# Training Function
def train(model, iterator, optimizer, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.train() # Set model to training mode

    for batch_idx, (features, labels) in enumerate(iterator):
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() # Clear gradients
        predictions = model(features).squeeze(1)
        loss = criterion(predictions, labels.float())
        acc = binary_accuracy(predictions, labels.float())

        loss.backward() # Backpropagation
        optimizer.step() # Update weights

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

# Evaluation Function
def evaluate(model, iterator, criterion):
    epoch_loss = 0
    epoch_acc = 0
    all_preds = []
    all_labels = []

    model.eval() # Set model to evaluation mode
    with torch.no_grad(): # Disable gradient calculation
        for batch_idx, (features, labels) in enumerate(iterator):
            features = features.to(device)
            labels = labels.to(device)

            predictions = model(features).squeeze(1)
            loss = criterion(predictions, labels.float())
            acc = binary_accuracy(predictions, labels.float())

            epoch_loss += loss.item()
            epoch_acc += acc.item()

            rounded_preds = torch.round(torch.sigmoid(predictions))
            all_preds.extend(rounded_preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return epoch_loss / len(iterator), epoch_acc / len(iterator), all_preds, all_labels

# Training Loop
N_EPOCHS = 10

best_valid_loss = float('inf')

print(f"Training model on {device}...")

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    valid_loss, valid_acc, _, _ = evaluate(model, test_loader, criterion) # Use test_loader as validation here

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best_model.pt')

    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'         | Test Loss: {valid_loss:.3f} | Test Acc: {valid_acc*100:.2f}%')

# Load the best model and evaluate on the test set
model.load_state_dict(torch.load('best_model.pt'))
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)

print(f'\nFinal Test Loss: {test_loss:.3f} | Final Test Acc: {test_acc*100:.2f}%')

# Calculate additional metrics
precision, recall, f1_score, _ = precision_recall_fscore_support(test_labels, test_preds, average='binary', zero_division=0)

print(f"\nPrecision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1_score:.3f}")

Training model on cpu...
Epoch: 01 | Train Loss: 0.298 | Train Acc: 88.57%
         | Test Loss: 0.159 | Test Acc: 93.72%
Epoch: 02 | Train Loss: 0.123 | Train Acc: 96.09%
         | Test Loss: 0.108 | Test Acc: 96.44%
Epoch: 03 | Train Loss: 0.070 | Train Acc: 98.10%
         | Test Loss: 0.081 | Test Acc: 97.57%
Epoch: 04 | Train Loss: 0.043 | Train Acc: 98.82%
         | Test Loss: 0.078 | Test Acc: 97.57%
Epoch: 05 | Train Loss: 0.028 | Train Acc: 99.35%
         | Test Loss: 0.075 | Test Acc: 97.83%
Epoch: 06 | Train Loss: 0.018 | Train Acc: 99.60%
         | Test Loss: 0.068 | Test Acc: 98.18%
Epoch: 07 | Train Loss: 0.012 | Train Acc: 99.84%
         | Test Loss: 0.084 | Test Acc: 97.57%
Epoch: 08 | Train Loss: 0.010 | Train Acc: 99.84%
         | Test Loss: 0.070 | Test Acc: 98.18%
Epoch: 09 | Train Loss: 0.007 | Train Acc: 99.87%
         | Test Loss: 0.071 | Test Acc: 98.35%
Epoch: 10 | Train Loss: 0.006 | Train Acc: 99.89%
         | Test Loss: 0.077 | Test Acc: 98.18%

Fina

In [2]:
import pandas as pd
import requests
import io

# URL for the SMS Spam Collection dataset (UCI Machine Learning Repository)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

# Download the zip file
r = requests.get(url, allow_redirects=True)

# Extract the content of the zip file (there's usually one .tsv file inside)
# Using io.BytesIO to treat the downloaded content as a file
import zipfile
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    with z.open('SMSSpamCollection') as f:
        df = pd.read_csv(f, sep='\t', header=None, names=['label', 'message'])

print("Dataset loaded successfully.")
display(df.head())

Dataset loaded successfully.


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
display(df['label'].value_counts())

,count
label,
ham,4825
spam,747
